In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("esquema", "bronze")

dbutils.widgets.text("tabla_source_sql", "customers")
dbutils.widgets.text("tabla_bronze", "customers_bronze")

catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
tabla_source_sql = dbutils.widgets.get("tabla_source_sql")
tabla_bronze = dbutils.widgets.get("tabla_bronze")

In [0]:
jdbc_url = (
    "jdbc:sqlserver://server-az-sql-db-renzocavero.database.windows.net:1433;"
    "database=az-sql-db-renzocavero;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

connection_properties = {
    "user": dbutils.secrets.get(scope="kv-scope", key="sql-username"),
    "password": dbutils.secrets.get(scope="kv-scope", key="sql-pw"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
raw_df = spark.read.jdbc(
    url=jdbc_url,
    table=f"dbo.{tabla_source_sql}",
    properties=connection_properties
)

In [0]:
from pyspark.sql.functions import trim, col, current_timestamp
from pyspark.sql.types import StringType

str_cols = [
    field.name
    for field in raw_df.schema.fields
    if isinstance(field.dataType, StringType)
]

bronze_df = raw_df

for c in str_cols:
    bronze_df = bronze_df.withColumn(c, trim(col(c)))

bronze_df = bronze_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalogo}.{esquema}.{tabla_bronze}")

In [0]:
bronze_df.printSchema()
bronze_df.limit(5).display()
